# Proyecto de Inteligencia Artificial  
## ETL para predicción de incumplimiento crediticio

**Fuente del dataset declarada por el equipo:** *AmExpert 2021 CODELAB - Machine Learning Hackathon*.

Este notebook documenta el avance correspondiente a la etapa de **ETL (Extracción, Transformación y Carga)**. El objetivo es preparar el conjunto de datos antes de comparar los modelos **Random Forest** y **XGBoost**.

## 1. Librerías y configuración

El notebook busca primero un archivo llamado `datos_credito.csv`. También reconoce el nombre original cargado en este proyecto y las rutas comunes de Jupyter, Google Colab y ChatGPT.


In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

DIRECTORIO_SALIDA = Path("salidas_etl")
DIRECTORIO_SALIDA.mkdir(exist_ok=True)

NOMBRES_CSV = [
    "datos_credito.csv",
    "81fff036-c2b8-4de4-8e08-196e55d9414a.csv",
]

RUTAS_CANDIDATAS = []
for nombre in NOMBRES_CSV:
    RUTAS_CANDIDATAS.extend([
        Path(nombre),
        Path("/content") / nombre,
        Path("/mnt/data") / nombre,
    ])

ARCHIVO_CSV = next((ruta for ruta in RUTAS_CANDIDATAS if ruta.exists()), None)

if ARCHIVO_CSV is None:
    csv_disponibles = [
        ruta for ruta in Path.cwd().glob("*.csv")
        if "limpios" not in ruta.name
        and "modelado" not in ruta.name
        and "reporte" not in ruta.name
        and "diccionario" not in ruta.name
    ]
    if len(csv_disponibles) == 1:
        ARCHIVO_CSV = csv_disponibles[0]
    else:
        raise FileNotFoundError(
            "No se encontró el dataset. Coloque el CSV junto al notebook "
            "con el nombre 'datos_credito.csv'."
        )

print(f"Archivo seleccionado: {ARCHIVO_CSV.resolve()}")


Archivo seleccionado: /mnt/data/datos_credito.csv


## 2. Extracción de datos

Se carga el CSV y se conserva una copia original para poder comparar la calidad de los datos antes y después del ETL.


In [2]:
df_original = pd.read_csv(ARCHIVO_CSV)
df = df_original.copy()

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
display(df.head())


Filas: 45,528
Columnas: 19


,customer_id,name,age,gender,owns_car,owns_house,no_of_children,net_yearly_income,no_of_days_employed,occupation_type,total_family_members,migrant_worker,yearly_debt_payments,credit_limit,credit_limit_used(%),credit_score,prev_defaults,default_in_last_6months,credit_card_default
0,CST_115179,ita Bose,46,F,N,Y,0.00,"107,934.04",612.00,Unknown,1.00,1.00,"33,070.28","18,690.93",73,544.00,2,1,1
1,CST_121920,Alper Jonathan,29,M,N,Y,0.00,"109,862.62","2,771.00",Laborers,2.00,0.00,"15,329.53","37,745.19",52,857.00,0,0,0
2,CST_109330,Umesh Desai,37,M,N,Y,0.00,"230,153.17",204.00,Laborers,2.00,0.00,"48,416.60","41,598.36",43,650.00,0,0,0
3,CST_128288,Rie,39,F,N,Y,0.00,"122,325.82","11,941.00",Core staff,2.00,0.00,"22,574.36","32,627.76",20,754.00,0,0,0
4,CST_151355,McCool,46,M,Y,Y,0.00,"387,286.00","1,459.00",Core staff,1.00,0.00,"38,282.95","52,950.64",75,927.00,0,0,0


## 3. Diagnóstico inicial

El diagnóstico permite identificar tipos de datos, valores nulos, duplicados y cantidad de categorías diferentes.


In [3]:
reporte_inicial = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=False),
}).sort_values("nulos", ascending=False)

print(f"Duplicados exactos: {df.duplicated().sum():,}")
print(f"IDs de cliente duplicados: {df.duplicated(subset=['customer_id']).sum():,}")
display(reporte_inicial)


Duplicados exactos: 0
IDs de cliente duplicados: 0


,tipo_dato,nulos,porcentaje_nulos,valores_unicos
no_of_children,float64,774,1.70,11
owns_car,object,547,1.20,3
no_of_days_employed,float64,463,1.02,7875
yearly_debt_payments,float64,95,0.21,45252
migrant_worker,float64,87,0.19,3
total_family_members,float64,83,0.18,11
credit_score,float64,8,0.02,451
age,int64,0,0.00,33
name,object,0,0.00,4010
customer_id,object,0,0.00,45528


## 4. Diccionarios de traducción

Los diccionarios centralizan la traducción de columnas y ocupaciones. Esto hace que el ETL sea trazable y fácil de modificar.


In [4]:
TRADUCCION_COLUMNAS = {
    "customer_id": "id_cliente",
    "name": "nombre",
    "age": "edad",
    "gender": "genero",
    "owns_car": "tiene_auto",
    "owns_house": "tiene_casa",
    "no_of_children": "numero_hijos",
    "net_yearly_income": "ingreso_anual_neto",
    "no_of_days_employed": "dias_empleado",
    "occupation_type": "tipo_ocupacion",
    "total_family_members": "total_miembros_familia",
    "migrant_worker": "trabajador_migrante",
    "yearly_debt_payments": "pagos_anuales_deuda",
    "credit_limit": "limite_credito",
    "credit_limit_used(%)": "porcentaje_limite_credito_utilizado",
    "credit_score": "puntaje_crediticio",
    "prev_defaults": "incumplimientos_previos",
    "default_in_last_6months": "incumplimiento_ultimos_6_meses",
    "credit_card_default": "incumplimiento_tarjeta_credito",
}

TRADUCCION_OCUPACIONES = {
    "Unknown": "Desconocido",
    "Laborers": "Obreros",
    "Sales staff": "Personal de ventas",
    "Core staff": "Personal administrativo",
    "Managers": "Gerentes",
    "Drivers": "Conductores",
    "High skill tech staff": "Personal técnico altamente calificado",
    "Accountants": "Contadores",
    "Medicine staff": "Personal médico",
    "Security staff": "Personal de seguridad",
    "Cooking staff": "Personal de cocina",
    "Cleaning staff": "Personal de limpieza",
    "Private service staff": "Personal de servicio privado",
    "Low-skill Laborers": "Trabajadores no calificados",
    "Waiters/barmen staff": "Meseros y cantineros",
    "Secretaries": "Secretarios",
    "Realty agents": "Agentes inmobiliarios",
    "HR staff": "Personal de recursos humanos",
    "IT staff": "Personal de tecnología de la información",
}

display(pd.DataFrame(
    TRADUCCION_COLUMNAS.items(),
    columns=["columna_original", "columna_espanol"]
))


,columna_original,columna_espanol
0,customer_id,id_cliente
1,name,nombre
2,age,edad
3,gender,genero
4,owns_car,tiene_auto
5,owns_house,tiene_casa
6,no_of_children,numero_hijos
7,net_yearly_income,ingreso_anual_neto
8,no_of_days_employed,dias_empleado
9,occupation_type,tipo_ocupacion


## 5. Transformación y limpieza

### Criterios aplicados
1. Se limpian espacios innecesarios en textos.
2. Se eliminan duplicados exactos y duplicados por `customer_id`.
3. Se traducen las columnas.
4. Se convierten las variables numéricas con `errors="coerce"`.
5. Los valores cercanos a **365,243 días de empleo** se consideran un marcador técnico inválido y se convierten temporalmente en nulos.
6. Los faltantes numéricos se imputan con mediana o moda.
7. Para `dias_empleado`, primero se utiliza la mediana de la ocupación y luego la mediana global.
8. Se crean indicadores de ausencia en la versión para modelado.


In [5]:
df = df_original.copy()
bitacora = []

filas_iniciales = len(df)
duplicados_exactos = int(df.duplicated().sum())
duplicados_id = int(df.duplicated(subset=["customer_id"]).sum())

# 1. Normalización de textos.
for columna in df.select_dtypes(include="object").columns:
    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# 2. Eliminación de duplicados.
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["customer_id"], keep="first").copy()

# 3. Traducción de columnas.
df = df.rename(columns=TRADUCCION_COLUMNAS)
df["nombre"] = df["nombre"].str.title()

# 4. Conversión numérica.
columnas_numericas = [
    "edad", "numero_hijos", "ingreso_anual_neto", "dias_empleado",
    "total_miembros_familia", "trabajador_migrante",
    "pagos_anuales_deuda", "limite_credito",
    "porcentaje_limite_credito_utilizado", "puntaje_crediticio",
    "incumplimientos_previos", "incumplimiento_ultimos_6_meses",
    "incumplimiento_tarjeta_credito",
]
for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

# 5. Tratamiento de valores inválidos.
mascara_dias_invalidos = (df["dias_empleado"] < 0) | (df["dias_empleado"] >= 365000)
dias_invalidos = int(mascara_dias_invalidos.sum())
df.loc[mascara_dias_invalidos, "dias_empleado"] = np.nan

df.loc[df["numero_hijos"] < 0, "numero_hijos"] = np.nan
df.loc[df["total_miembros_familia"] < 1, "total_miembros_familia"] = np.nan
df.loc[
    ~df["porcentaje_limite_credito_utilizado"].between(0, 100),
    "porcentaje_limite_credito_utilizado",
] = np.nan
df.loc[~df["trabajador_migrante"].isin([0, 1]), "trabajador_migrante"] = np.nan
df.loc[
    ~df["incumplimiento_ultimos_6_meses"].isin([0, 1]),
    "incumplimiento_ultimos_6_meses",
] = np.nan
df.loc[
    ~df["incumplimiento_tarjeta_credito"].isin([0, 1]),
    "incumplimiento_tarjeta_credito",
] = np.nan

# 6. Traducción de categorías.
df["genero"] = (
    df["genero"]
    .str.upper()
    .replace({"F": "Femenino", "M": "Masculino", "XNA": "No especificado"})
    .fillna("No especificado")
)

for columna in ["tiene_auto", "tiene_casa"]:
    df[columna] = (
        df[columna]
        .str.upper()
        .replace({"Y": "Sí", "N": "No"})
        .fillna("No informado")
    )

df["tipo_ocupacion"] = (
    df["tipo_ocupacion"]
    .replace(TRADUCCION_OCUPACIONES)
    .fillna("Desconocido")
)

display(df.head())


,id_cliente,nombre,edad,genero,tiene_auto,tiene_casa,numero_hijos,ingreso_anual_neto,dias_empleado,tipo_ocupacion,total_miembros_familia,trabajador_migrante,pagos_anuales_deuda,limite_credito,porcentaje_limite_credito_utilizado,puntaje_crediticio,incumplimientos_previos,incumplimiento_ultimos_6_meses,incumplimiento_tarjeta_credito
0,CST_115179,Ita Bose,46,Femenino,No,Sí,0.00,"107,934.04",612.00,Desconocido,1.00,1.00,"33,070.28","18,690.93",73.00,544.00,2,1.00,1.00
1,CST_121920,Alper Jonathan,29,Masculino,No,Sí,0.00,"109,862.62","2,771.00",Obreros,2.00,0.00,"15,329.53","37,745.19",52.00,857.00,0,0.00,0.00
2,CST_109330,Umesh Desai,37,Masculino,No,Sí,0.00,"230,153.17",204.00,Obreros,2.00,0.00,"48,416.60","41,598.36",43.00,650.00,0,0.00,0.00
3,CST_128288,Rie,39,Femenino,No,Sí,0.00,"122,325.82","11,941.00",Personal administrativo,2.00,0.00,"22,574.36","32,627.76",20.00,754.00,0,0.00,0.00
4,CST_151355,Mccool,46,Masculino,Sí,Sí,0.00,"387,286.00","1,459.00",Personal administrativo,1.00,0.00,"38,282.95","52,950.64",75.00,927.00,0,0.00,0.00


In [6]:
# Copia destinada al modelado: conserva los indicadores binarios como 0 y 1.
df_modelo = df.copy()

columnas_con_nulos = [
    "numero_hijos",
    "dias_empleado",
    "total_miembros_familia",
    "trabajador_migrante",
    "pagos_anuales_deuda",
    "puntaje_crediticio",
]

for columna in columnas_con_nulos:
    df_modelo[f"{columna}_era_nulo"] = df_modelo[columna].isna().astype("int8")

# Estadísticos para imputación.
mediana_hijos = df["numero_hijos"].median()
mediana_familia = df["total_miembros_familia"].median()
mediana_deuda = df["pagos_anuales_deuda"].median()
mediana_puntaje = df["puntaje_crediticio"].median()
moda_migrante = df["trabajador_migrante"].mode(dropna=True).iloc[0]

mediana_dias_por_ocupacion = df.groupby(
    "tipo_ocupacion"
)["dias_empleado"].transform("median")
mediana_dias_global = df["dias_empleado"].median()

for marco in [df, df_modelo]:
    marco["numero_hijos"] = marco["numero_hijos"].fillna(mediana_hijos).round()
    marco["dias_empleado"] = (
        marco["dias_empleado"]
        .fillna(mediana_dias_por_ocupacion)
        .fillna(mediana_dias_global)
        .round()
    )
    marco["total_miembros_familia"] = (
        marco["total_miembros_familia"].fillna(mediana_familia).round()
    )
    marco["trabajador_migrante"] = (
        marco["trabajador_migrante"].fillna(moda_migrante).round()
    )
    marco["pagos_anuales_deuda"] = (
        marco["pagos_anuales_deuda"].fillna(mediana_deuda)
    )
    marco["puntaje_crediticio"] = (
        marco["puntaje_crediticio"].fillna(mediana_puntaje).round()
    )

columnas_enteras = [
    "edad", "numero_hijos", "dias_empleado", "total_miembros_familia",
    "trabajador_migrante", "porcentaje_limite_credito_utilizado",
    "puntaje_crediticio", "incumplimientos_previos",
    "incumplimiento_ultimos_6_meses",
    "incumplimiento_tarjeta_credito",
]

for marco in [df, df_modelo]:
    for columna in columnas_enteras:
        marco[columna] = marco[columna].astype("int64")

# Versión legible: los indicadores binarios se muestran como Sí/No.
df_limpio = df.copy()
df_limpio["trabajador_migrante"] = df_limpio["trabajador_migrante"].map({1: "Sí", 0: "No"})
df_limpio["incumplimiento_ultimos_6_meses"] = df_limpio[
    "incumplimiento_ultimos_6_meses"
].map({1: "Sí", 0: "No"})
df_limpio["incumplimiento_tarjeta_credito"] = df_limpio[
    "incumplimiento_tarjeta_credito"
].map({1: "Sí", 0: "No"})

bitacora = pd.DataFrame([
    ["Filas iniciales", filas_iniciales],
    ["Duplicados exactos detectados", duplicados_exactos],
    ["IDs duplicados detectados", duplicados_id],
    ["Filas finales", len(df_limpio)],
    ["Valores técnicos de días de empleo corregidos", dias_invalidos],
    ["Nulos finales en versión limpia", int(df_limpio.isna().sum().sum())],
    ["Nulos finales en versión para modelado", int(df_modelo.isna().sum().sum())],
], columns=["proceso", "resultado"])

display(bitacora)


,proceso,resultado
0,Filas iniciales,45528
1,Duplicados exactos detectados,0
2,IDs duplicados detectados,0
3,Filas finales,45528
4,Valores técnicos de días de empleo corregidos,8098
5,Nulos finales en versión limpia,0
6,Nulos finales en versión para modelado,0


## 6. Validaciones automáticas

Las siguientes aserciones detienen el proceso cuando alguna condición importante no se cumple.


In [7]:
assert df_limpio["id_cliente"].is_unique, "Existen IDs duplicados."
assert df_limpio.isna().sum().sum() == 0, "La versión limpia todavía contiene nulos."
assert df_modelo.isna().sum().sum() == 0, "La versión para modelado todavía contiene nulos."
assert set(df_limpio["tiene_auto"].unique()).issubset({"Sí", "No", "No informado"})
assert set(df_limpio["tiene_casa"].unique()).issubset({"Sí", "No", "No informado"})
assert set(df_modelo["incumplimiento_tarjeta_credito"].unique()).issubset({0, 1})
assert df_limpio["porcentaje_limite_credito_utilizado"].between(0, 100).all()

print("Todas las validaciones fueron superadas correctamente.")
display(df_limpio.head(10))


Todas las validaciones fueron superadas correctamente.


,id_cliente,nombre,edad,genero,tiene_auto,tiene_casa,numero_hijos,ingreso_anual_neto,dias_empleado,tipo_ocupacion,total_miembros_familia,trabajador_migrante,pagos_anuales_deuda,limite_credito,porcentaje_limite_credito_utilizado,puntaje_crediticio,incumplimientos_previos,incumplimiento_ultimos_6_meses,incumplimiento_tarjeta_credito
0,CST_115179,Ita Bose,46,Femenino,No,Sí,0,"107,934.04",612,Desconocido,1,Sí,"33,070.28","18,690.93",73,544,2,Sí,Sí
1,CST_121920,Alper Jonathan,29,Masculino,No,Sí,0,"109,862.62",2771,Obreros,2,No,"15,329.53","37,745.19",52,857,0,No,No
2,CST_109330,Umesh Desai,37,Masculino,No,Sí,0,"230,153.17",204,Obreros,2,No,"48,416.60","41,598.36",43,650,0,No,No
3,CST_128288,Rie,39,Femenino,No,Sí,0,"122,325.82",11941,Personal administrativo,2,No,"22,574.36","32,627.76",20,754,0,No,No
4,CST_151355,Mccool,46,Masculino,Sí,Sí,0,"387,286.00",1459,Personal administrativo,1,No,"38,282.95","52,950.64",75,927,0,No,No
5,CST_123268,Sarah Marsh,46,Femenino,Sí,No,0,"252,765.91",2898,Contadores,2,Sí,"37,046.86","40,245.64",19,937,0,No,No
6,CST_127502,Mason,38,Masculino,No,Sí,1,"262,389.20",5541,Personal técnico altamente calificado,3,No,"50,839.39","41,311.08",42,733,0,No,No
7,CST_151722,Saba,46,Femenino,Sí,Sí,1,"241,211.39",1448,Personal administrativo,3,No,"30,008.46","32,209.22",91,906,0,No,No
8,CST_133768,Ashutosh,40,Femenino,No informado,Sí,0,"210,091.43",11551,Obreros,2,No,"21,521.89","65,037.74",14,783,0,No,No
9,CST_111670,David Milliken,39,Femenino,Sí,Sí,2,"207,109.13",2791,Personal técnico altamente calificado,4,No,"9,509.10","28,425.52",14,666,0,No,No


## 7. Distribución de la variable objetivo

La variable objetivo está desbalanceada. Este resultado será importante cuando se entrenen Random Forest y XGBoost, porque no bastará con medir solamente exactitud; también deberán revisarse **Recall, F1, AUC-ROC y matriz de confusión**.


In [8]:
distribucion_objetivo = (
    df_modelo["incumplimiento_tarjeta_credito"]
    .value_counts()
    .sort_index()
    .rename(index={0: "No moroso", 1: "Moroso"})
    .to_frame("cantidad")
)

distribucion_objetivo["porcentaje"] = (
    distribucion_objetivo["cantidad"] / distribucion_objetivo["cantidad"].sum() * 100
).round(2)

display(distribucion_objetivo)


,cantidad,porcentaje
incumplimiento_tarjeta_credito,,
No moroso,41831,91.88
Moroso,3697,8.12


## 8. Exportación de resultados

Se generan cuatro archivos:

- `clientes_credito_limpios.csv`: versión legible en español.
- `clientes_credito_para_modelado.csv`: mantiene la variable objetivo y los indicadores binarios como 0/1.
- `reporte_calidad_datos.csv`: compara la calidad antes y después.
- `diccionario_datos.csv`: explica la traducción de cada variable.


In [9]:
ruta_limpio = DIRECTORIO_SALIDA / "clientes_credito_limpios.csv"
ruta_modelo = DIRECTORIO_SALIDA / "clientes_credito_para_modelado.csv"
ruta_reporte = DIRECTORIO_SALIDA / "reporte_calidad_datos.csv"
ruta_diccionario = DIRECTORIO_SALIDA / "diccionario_datos.csv"

df_limpio.to_csv(ruta_limpio, index=False, encoding="utf-8-sig")
df_modelo.to_csv(ruta_modelo, index=False, encoding="utf-8-sig")

reporte_antes = pd.DataFrame({
    "columna_original": df_original.columns,
    "tipo_dato_original": [str(tipo) for tipo in df_original.dtypes],
    "nulos_antes": df_original.isna().sum().values,
    "porcentaje_nulos_antes": (df_original.isna().mean().values * 100).round(2),
    "valores_unicos_antes": [
        df_original[columna].nunique(dropna=False)
        for columna in df_original.columns
    ],
})

reporte_despues = pd.DataFrame({
    "columna_espanol": df_limpio.columns,
    "tipo_dato_final": [str(tipo) for tipo in df_limpio.dtypes],
    "nulos_despues": df_limpio.isna().sum().values,
    "valores_unicos_despues": [
        df_limpio[columna].nunique(dropna=False)
        for columna in df_limpio.columns
    ],
})

reporte_calidad = pd.concat([reporte_antes, reporte_despues], axis=1)
reporte_calidad.to_csv(ruta_reporte, index=False, encoding="utf-8-sig")

descripciones = {
    "id_cliente": "Identificador único del cliente.",
    "nombre": "Nombre del cliente.",
    "edad": "Edad del cliente en años.",
    "genero": "Género reportado por el cliente.",
    "tiene_auto": "Indica si el cliente posee automóvil.",
    "tiene_casa": "Indica si el cliente posee vivienda.",
    "numero_hijos": "Cantidad de hijos.",
    "ingreso_anual_neto": "Ingreso anual neto del cliente.",
    "dias_empleado": "Cantidad de días de empleo.",
    "tipo_ocupacion": "Ocupación del cliente traducida al español.",
    "total_miembros_familia": "Total de miembros del grupo familiar.",
    "trabajador_migrante": "Indica si la persona es trabajador migrante.",
    "pagos_anuales_deuda": "Monto anual destinado al pago de deudas.",
    "limite_credito": "Límite de crédito disponible.",
    "porcentaje_limite_credito_utilizado": "Porcentaje del límite utilizado.",
    "puntaje_crediticio": "Puntaje crediticio.",
    "incumplimientos_previos": "Cantidad de incumplimientos anteriores.",
    "incumplimiento_ultimos_6_meses": "Incumplimiento en los últimos seis meses.",
    "incumplimiento_tarjeta_credito": "Variable objetivo de incumplimiento.",
}

diccionario_datos = pd.DataFrame([
    {
        "columna_original": original,
        "columna_espanol": espanol,
        "descripcion": descripciones[espanol],
    }
    for original, espanol in TRADUCCION_COLUMNAS.items()
])

diccionario_datos.to_csv(ruta_diccionario, index=False, encoding="utf-8-sig")

print("Archivos generados:")
for ruta in [ruta_limpio, ruta_modelo, ruta_reporte, ruta_diccionario]:
    print(f" - {ruta.resolve()}")


Archivos generados:
 - /mnt/data/salidas_etl/clientes_credito_limpios.csv
 - /mnt/data/salidas_etl/clientes_credito_para_modelado.csv
 - /mnt/data/salidas_etl/reporte_calidad_datos.csv
 - /mnt/data/salidas_etl/diccionario_datos.csv


## 9. Carga opcional a SQL Server con Docker

El archivo `docker-compose.yml` levanta SQL Server en el puerto `1433`.

### Comandos

```bash
docker compose up -d
docker ps
```

Antes de ejecutar la siguiente celda:

1. Copiar `.env.example` como `.env`.
2. Cambiar la contraseña.
3. Instalar **Microsoft ODBC Driver 18 for SQL Server**.
4. Instalar dependencias: `pip install sqlalchemy pyodbc python-dotenv`.
5. Cambiar `CARGAR_A_SQL_SERVER` a `True`.

La contraseña no se escribe directamente en el notebook.


In [10]:
CARGAR_A_SQL_SERVER = False

if not CARGAR_A_SQL_SERVER:
    print(
        "Carga omitida. Cambie CARGAR_A_SQL_SERVER a True "
        "cuando el contenedor esté activo."
    )
else:
    from dotenv import load_dotenv
    import pyodbc
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL

    load_dotenv()

    servidor = os.getenv("SQL_SERVER_HOST", "localhost")
    puerto = int(os.getenv("SQL_SERVER_PORT", "1433"))
    base_datos = os.getenv("SQL_SERVER_DATABASE", "riesgo_crediticio")
    usuario = os.getenv("SQL_SERVER_USER", "sa")
    contrasena = os.getenv("MSSQL_SA_PASSWORD")

    if not contrasena:
        raise ValueError(
            "No se encontró MSSQL_SA_PASSWORD. Revise el archivo .env."
        )

    controladores = pyodbc.drivers()
    controlador = next(
        (
            item for item in controladores
            if "ODBC Driver 18 for SQL Server" in item
        ),
        None,
    )
    if controlador is None:
        raise RuntimeError(
            "No está instalado ODBC Driver 18 for SQL Server. "
            f"Controladores disponibles: {controladores}"
        )

    url_master = URL.create(
        "mssql+pyodbc",
        username=usuario,
        password=contrasena,
        host=servidor,
        port=puerto,
        database="master",
        query={
            "driver": controlador,
            "TrustServerCertificate": "yes",
        },
    )

    motor_master = create_engine(
        url_master,
        isolation_level="AUTOCOMMIT",
        fast_executemany=True,
    )

    with motor_master.connect() as conexion:
        existe = conexion.execute(
            text("SELECT DB_ID(:nombre)"),
            {"nombre": base_datos},
        ).scalar()

        if existe is None:
            if not base_datos.replace("_", "").isalnum():
                raise ValueError("Nombre de base de datos no permitido.")
            conexion.execute(text(f"CREATE DATABASE [{base_datos}]"))
            print(f"Base de datos creada: {base_datos}")
        else:
            print(f"La base de datos ya existe: {base_datos}")

    url_bd = URL.create(
        "mssql+pyodbc",
        username=usuario,
        password=contrasena,
        host=servidor,
        port=puerto,
        database=base_datos,
        query={
            "driver": controlador,
            "TrustServerCertificate": "yes",
        },
    )

    motor_bd = create_engine(url_bd, fast_executemany=True)

    df_limpio.to_sql(
        "clientes_credito_limpios",
        con=motor_bd,
        if_exists="replace",
        index=False,
        chunksize=1000,
    )

    df_modelo.to_sql(
        "clientes_credito_para_modelado",
        con=motor_bd,
        if_exists="replace",
        index=False,
        chunksize=1000,
    )

    with motor_bd.connect() as conexion:
        filas_limpias = conexion.execute(
            text("SELECT COUNT(*) FROM clientes_credito_limpios")
        ).scalar()
        filas_modelo = conexion.execute(
            text("SELECT COUNT(*) FROM clientes_credito_para_modelado")
        ).scalar()

    print(f"Filas en clientes_credito_limpios: {filas_limpias:,}")
    print(f"Filas en clientes_credito_para_modelado: {filas_modelo:,}")


Carga omitida. Cambie CARGAR_A_SQL_SERVER a True cuando el contenedor esté activo.
